# Experiment

## Import libraries

In [11]:
import pandas as pd

file_path = "iir_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="date_str",
    value_name="value",
)

# Filter only valid dd/mm/yyyy
df = df[df["date_str"].str.match(r"\d{2}/\d{2}/\d{4}")]

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Convert to datetime
df["date"] = pd.to_datetime(df["date_str"], format="%d/%m/%Y", errors="coerce")

# Extract year, month, day
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# Pivot table to wide format
df = df.pivot_table(
    index=["year", "month", "day"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by date
df = df.sort_values(["year", "month", "day"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

# Rename columns
df.rename(
    columns={
        "1_months": "_1_months",
        "1_weeks": "_1_weeks",
        "2_weeks": "_2_weeks",
        "3_months": "_3_months",
        "6_months": "_6_months",
        "9_months": "_9_months",
    },
    inplace=True,
)

df

Chỉ tiêu,year,month,day,_1_months,_1_weeks,_2_weeks,_3_months,_6_months,_9_months,overnight
0,2020,8,12,0.56,0.26,0.48,1.56,2.57,4.50,0.19
1,2020,8,13,0.57,0.31,0.32,1.08,3.18,4.50,0.18
2,2020,8,14,0.72,0.24,0.30,1.57,3.70,4.33,0.18
3,2020,8,17,0.50,0.25,0.31,2.02,3.90,4.33,0.19
4,2020,8,18,0.52,0.26,0.42,1.47,3.10,4.33,0.20
...,...,...,...,...,...,...,...,...,...,...
1241,2025,8,1,4.51,5.35,5.07,5.46,5.10,5.44,5.36
1242,2025,8,4,4.82,5.27,5.26,5.13,5.50,5.44,5.22
1243,2025,8,5,5.24,5.56,5.57,5.36,5.36,5.44,5.48
1244,2025,8,6,5.26,5.57,5.56,5.51,5.98,5.44,5.48


In [12]:
df.columns

Index(['year', 'month', 'day', '_1_months', '_1_weeks', '_2_weeks',
       '_3_months', '_6_months', '_9_months', 'overnight'],
      dtype='object', name='Chỉ tiêu')